In [1]:
import numpy as np

def fractional_accumulate(x0, r):
    """
    复现论文中的分数阶累加生成操作 (Fractional Order Accumulation)
    
    参数:
    x0 : array_like, 原始的一维非负数据序列 X^(0)
    r  : float, 累加阶数 (r > 0)
    
    返回:
    xr : ndarray, r阶累加后的序列 X^(r)
    """
    x0 = np.asarray(x0, dtype=float)
    n = len(x0)
    
    # 边界情况：当 r=0 时，累加序列就是原序列
    if r == 0:
        return x0.copy()
    
    # 边界情况：当 r=1 时，退化为传统的一阶累加 (GM(1,1)的标准操作)
    if r == 1.0:
        return np.cumsum(x0)
    
    # 1. 迭代计算广义二项式系数向量 (权重 C_m)
    # C_m = (r + m - 1) / m * C_{m-1}
    weights = np.ones(n)
    for m in range(1, n):
        weights[m] = weights[m-1] * (r + m - 1) / m
        
    # 2. 利用卷积计算累加和
    # 公式 sum_{i=1}^k C_{k-i} * x0(i) 本质上就是 x0 和 weights 的一维因果卷积
    xr = np.convolve(x0, weights, mode='full')[:n]
    
    return xr

# ==========================================
# 测试用例 (Test Cases)
# ==========================================
if __name__ == "__main__":
    # 假设有一组原始数据序列
    data_x0 = np.array([10.0, 12.0, 15.0, 18.0, 22.0])
    
    print("原始序列 X(0):")
    print(data_x0)
    
    print("\n1. 传统的 1 阶累加 (r=1) - 验证代码是否等价于 np.cumsum:")
    print("代码结果:", fractional_accumulate(data_x0, 1.0))
    print("Numpy原生:", np.cumsum(data_x0))
    
    print("\n2. 论文中的分数阶累加 (例如 r=0.5):")
    # r越小，旧数据的权重衰减越快，这体现了论文中提到的“新信息优先”原则
    print("r=0.5 结果:", fractional_accumulate(data_x0, 0.5))
    
    print("\n3. 论文中的分数阶累加 (例如 r=1.2):")
    # r越大(>1)，旧数据被赋予了比新数据更高的权重 (适用于长记忆过程)
    print("r=1.2 结果:", fractional_accumulate(data_x0, 1.2))

原始序列 X(0):
[10. 12. 15. 18. 22.]

1. 传统的 1 阶累加 (r=1) - 验证代码是否等价于 np.cumsum:
代码结果: [10. 22. 37. 55. 77.]
Numpy原生: [10. 22. 37. 55. 77.]

2. 论文中的分数阶累加 (例如 r=0.5):
r=0.5 结果: [10.       17.       24.75     33.125    43.109375]

3. 论文中的分数阶累加 (例如 r=1.2):
r=1.2 结果: [10.   24.   42.6  65.92 95.08]


In [19]:
import sys
import os

# 获取目标文件夹的绝对路径（相对于当前 notebook 的位置）
# 这里 '../utils' 表示上一级目录下的 utils 文件夹
target_path = os.path.abspath('../utils')

# 将该路径加入到 Python 的搜索路径中
if target_path not in sys.path:
    sys.path.append(target_path)

from CommonOperations.FractionalAccumulation import fago

x11 = fago(data_x0,0.5)
x12 = fago(data_x0,1.2)

print(x11)
print(x12)

[10.       17.       24.75     33.125    43.109375]
[10.   24.   42.6  65.92 95.08]


In [21]:
r_invteral = np.linspace(-2,2,100)


mse_list = []

for i in range(100):
    x0 = np.random.rand(10)+1
    
    xr1 = fractional_accumulate(x0,r_invteral[i])
    xr2 = fago(x0,r_invteral[i])
    
    mse = np.mean((xr1-xr2)**2)
    mse_list.append(mse)
    
    
print(mse_list)

print('最大误差(MSE):',max(mse_list))
print('最小误差(MSE):',min(mse_list))

    
    

[4.930380657631324e-33, 4.930380657631324e-32, 2.465190328815662e-32, 2.958228394578794e-32, 1.479114197289397e-32, 1.479114197289397e-32, 3.0814879110195774e-32, 2.958228394578794e-32, 2.958228394578794e-32, 9.860761315262648e-33, 1.1093356479670479e-32, 3.944304526105059e-32, 3.0814879110195774e-32, 1.6023737137301803e-32, 3.0814879110195774e-32, 9.860761315262648e-33, 2.958228394578794e-32, 0.0, 9.860761315262648e-33, 1.7256332301709632e-32, 4.930380657631324e-32, 4.930380657631324e-33, 2.465190328815662e-32, 1.479114197289397e-32, 4.4373425918681915e-32, 4.930380657631324e-33, 9.860761315262648e-33, 4.930380657631324e-33, 1.479114197289397e-32, 6.162975822039155e-33, 1.479114197289397e-32, 1.1093356479670479e-32, 1.479114197289397e-32, 9.860761315262648e-33, 1.6023737137301803e-32, 7.395570986446985e-33, 1.1093356479670479e-32, 1.232595164407831e-33, 4.930380657631324e-33, 2.465190328815662e-33, 3.697785493223493e-33, 0.0, 2.465190328815662e-33, 2.465190328815662e-33, 0.0, 3.697785